In [1]:
%gui qt
import napari
import numpy as np
import tifffile
import scipy as sp
import cv2
import scipy.ndimage as ndimage
from skimage.transform import rescale, resize, downscale_local_mean
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

In [2]:
viewer = napari.Viewer()

In [5]:
cats = {'Type':['Other', 'Chromatoid', 'Secretory', 'Gut', 'Epithelial', 'Junk', 'Doublet', 
                                ], 'Sample':['control', '4dpi', '8dpi', '14dpi'], 
                        'Annotator':['MEL', 'STN', 'MOL']}
my_cats = {'Dataset':['48hpa', 'control', '4dpi', '8dpi', '14dpi'], 
                        'Annotator':['MEL', 'STN', 'MOL'],
          'Type':['Chromatoid', 'Epidermal', 'Gut', 'Secretory', 'Proximal', 'Unknown', 'Junk', 'Other']}

In [6]:
datas = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210909_MER_20-15-5B_4dpi/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210129_MER_20-15-7b_8dpi_stack2/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210326_MER_20-15-8a_14dpi/cjw/DeepLearn/',
        'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20220107_MER_IMARE-110171_19_55_48hpa_C4/cjw/gcs_etomo/DeepLearn/']
datas_nucleoli = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn_Nucleoli/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210909_MER_20-15-5B_4dpi/cjw/DeepLearn_Nucleoli/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210129_MER_20-15-7b_8dpi_stack2/cjw/DeepLearn_Nucleoli/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210326_MER_20-15-8a_14dpi/cjw/DeepLearn_Nucleoli/']

data_fix = ['control', '4dpi', '8dpi', '14dpi', '48hpa']

# Load classification data

### Setup the list of data files to load

Original annotated datasets

In [7]:
csvs = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MEL_Split_20211101.csv',
       'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MOL_Split_20211106.csv',
       'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/STN_Split_20211103.csv']

Re-classifying the "others" from above

In [8]:
new_csvs = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MOL_Others_FINISHED.csv',
           'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/STN_Others_20220127_FINAL.csv', 
           'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MEL_Others_20211203_FINAL.csv']

Labeling the "amputated" dataset that came later.  Also has a mix of some of the original thrown in to see how consistent labeling is

In [9]:
amputated_csvs = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MEL_Amputated_List_final.csv',
           'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MOL_Amputated_List_20220406_FINAL.csv', 
           'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/STN_Amputated_List_20220325_FINAL.csv']

### Load the originals into df

In [10]:
df = pd.DataFrame()
for f in csvs:
    tmp = pd.read_csv(f)
    tmp['Annotator'] = f
    df = pd.concat([df, tmp])
df['Annotator'] = df['Annotator'].str.split('/').str[-1].str.split('_').str[0]
#pd.read_csv('S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Classifier/MEL_20211027.csv')

df['Dataset'] = df['File'].str.split('_').str[-1].str.split('.').str[0]
df['Id'] = df['File'].str.split('_').str[-2].str.split('/').str[-1]


### Merge with the relabeled "others" dataset

In [11]:
ndf = pd.DataFrame()
for f in new_csvs:
    tmp = pd.read_csv(f)
    tmp['Annotator'] = f
    ndf = pd.concat([ndf, tmp])
ndf['Annotator'] = ndf['Annotator'].str.split('/').str[-1].str.split('_').str[0]

ndf['Dataset'] = ndf['File'].str.split('_').str[-1].str.split('.').str[0]
ndf['Id'] = ndf['File'].str.split('_').str[-2].str.split('/').str[-1]

In [12]:
tdf = df.merge(ndf, on=['Dataset', 'Id'], how='left')

In [13]:
tdf['Type'] = tdf['Type_x']
tdf['Annotator'] = tdf['Annotator_x']

In [14]:
tdf.loc[~tdf['Annotator_y'].isna(), 'Type'] = tdf.loc[~tdf['Annotator_y'].isna(), 'Type_y']
tdf.loc[~tdf['Annotator_y'].isna(), 'Annotator'] = tdf.loc[~tdf['Annotator_y'].isna(), 'Annotator_y']

In [15]:
df = tdf

### Load the amputated dataset and merge.

Load the amputated csvs

In [16]:
adf = pd.DataFrame()
for f in amputated_csvs:
    tmp = pd.read_csv(f)
    tmp['Annotator'] = f
    adf = pd.concat([adf, tmp])
adf['Annotator'] = adf['Annotator'].str.split('/').str[-1].str.split('_').str[0]

adf['Dataset'] = adf['File'].str.split('_').str[-1].str.split('.').str[0]
adf['Id'] = adf['File'].str.split('_').str[-2].str.split('/').str[-1]
adf.loc[adf['Type']=='Other', 'Type'] = 'Unknown'
adf.loc[adf['Dataset']=='amputated', 'Dataset'] = '48hpa'
adf.loc[adf['Type']=='Epithelial', 'Type'] = 'Epidermal'

First split into two dataframes, those with relabeled, and those with new amputated labels

In [17]:
amp_df = adf[adf['Dataset']=='48hpa']
relab_df = adf[adf['Dataset']=='control']

In [18]:
df = df.rename(columns={'File_x':'File'})[['Type', 'File', 'Annotator', 'Dataset', 'Id']]

For now we'll just worry about getting the amputated in and check the relabels later

In [19]:
df = pd.concat([df, amp_df])

### Add some extra columns that are useful

Create new column that we can sum to use as fraction for that dataset. 
Also rename epithelial to Epidermal 

In [20]:
df = df.merge(df.groupby(['Dataset']).agg({'File':len}).reset_index(), on='Dataset').rename(columns={'File_x':'File', 
                                                                                                'File_y':'Fraction'})
df['Fraction'] = 1/df['Fraction']
df.loc[df['Type']=='Epithelial', 'Type'] = 'Epidermal'

Now load the center of mass/volume dataset from 20221107 and merge with df

In [21]:
loc_df = pd.read_csv('Volume-CMs.csv')

In [22]:
loc_df['Id'] = loc_df['ID'].astype(str).str.zfill(4)

In [23]:
df = df.merge(loc_df, on=['Dataset', 'Id'])

# Plot results

### Population fraction for each dataset

In [24]:
tdf = df.copy()
tdf.loc[tdf['Type'].isin(['ChromatoidDoublet', 'Distal', 'Doublet', 'Muscle', 'Progenitor', 'Neuronal']), 'Type'] = 'Other'
f = px.bar(tdf.groupby(['Type', 'Dataset']).agg({'Fraction':np.sum}).reset_index(), x='Type', y='Fraction', color='Dataset', 
      height=800, category_orders=my_cats, barmode='group')
f.write_html('TypeFraction.html')
f

### Labeling consistency investigation

In [25]:
merged_df = relab_df.merge(df, on=['Dataset', 'Id'])

In [26]:
merged_df[merged_df['Type_x']!=merged_df['Type_y']].pivot_table(index=['Annotator_x', 'Annotator_y'], values='Type_x', aggfunc=len)

Type_x
Annotator_x Annotator_y        
MEL         MOL               4
            STN              17
MOL         MEL               5
            MOL               5
            STN               1
STN         MEL              12
            STN               7

In [27]:
px.histogram(merged_df, x='Annotator_y', color='Type_y')

Appears STN is more reserved in calling and defaults to unknown more frequently.

Let's merge unpopular types together into "Other"

In [28]:
merged_df.loc[merged_df['Type_x'].isin(['ChromatoidDoublet', 'Distal', 'Doublet', 'Muscle', 'Progenitor', 'Neuronal']), 'Type_x'] = 'Other'
merged_df.loc[merged_df['Type_y'].isin(['ChromatoidDoublet', 'Distal', 'Doublet', 'Muscle', 'Progenitor', 'Neuronal']), 'Type_y'] = 'Other'


In [29]:
merged_pivot = merged_df.pivot_table(index='Type_x', columns='Type_y', values='Id', aggfunc=len).fillna(0)

In [30]:
f = px.imshow(merged_pivot, color_continuous_scale=[(0, "blue"), (0.05, "red"), (0.2, 'green'), (1, "yellow")], 
          title='11% are inconsistently classified but most are deemed unknown by one analyst')
f.write_html('ConfusionMatrix_wUnknown.html')
f

Almost all of the inconsistencies have Unknown as one of the classifications.  This is not so much measuring error as it is confidence, so let's look at just those that were classified by both with confidence.

In [31]:
f = px.imshow(merged_pivot.iloc[0:-2,0:-2], color_continuous_scale=[(0, "blue"), (0.05, "red"), (0.2, 'green'), (1, "yellow")],
         title='Only 2.5% of those confidently identified by both are misclassified')
f.write_html('ConfusionMatrix_woUnknown.html')
f

In [58]:
print([len(merged_df[merged_df['Type_x']!=merged_df['Type_y']]), len(merged_df)])
len(merged_df[merged_df['Type_x']!=merged_df['Type_y']])/len(merged_df)

[50, 448]


0.11160714285714286

So if we look at all of them, 11% had different classifications between the two annotators.  However MOST of these are from one person saying they couldn't tell and the other having a guess, let's look at just the ones where both were certain.

In [33]:
not_unknown = merged_df[(merged_df['Type_x']!='Unknown') & (merged_df['Type_y']!='Unknown')]

In [59]:
print([len(not_unknown[not_unknown['Type_x']!=not_unknown['Type_y']]),len(not_unknown)])
len(not_unknown[not_unknown['Type_x']!=not_unknown['Type_y']])/len(not_unknown)

[6, 267]


0.02247191011235955

In cases where both annotators tried to classify, only 3% were inconsistent.

### Export data for Steph

In [47]:
tdf.columns

Index(['Type', 'File', 'Annotator', 'Dataset', 'Id', ' ', 'Fraction',
       'Unnamed: 0', 'Path', 'ID', 'Volume', 'Z', 'Y', 'X'],
      dtype='object')

Export csvs for Steph to look at

In [48]:
tdf['Volume(um^3)'] = tdf['Volume'] * 0.06 * 0.06 * 0.06
merged_df.to_csv('InconsistentClassifications.csv', index=False)
not_unknown.to_csv('InconsistentClassifications_woUnknown.csv', index=False)
tdf.to_csv('TypeFraction.csv', index=False)

Look at the amputated/reannotated set to see if they were all neoblasts

In [52]:
adf.pivot_table(index='Dataset', values='Id', aggfunc=len)

,Id
Dataset,
48hpa,2534
control,448


In [54]:
print(448/len(adf))

0.15023474178403756


In [55]:
#adf.groupby(['Dataset', 'Type']).agg({'Id':len}).reset_index()

Get simple numbers

In [46]:
tdf.groupby(['Dataset']).agg({'Fraction':np.sum, 'ID':len}).reset_index()

,Dataset,Fraction,ID
0,14dpi,1.0,1472
1,48hpa,1.0,2534
2,4dpi,1.0,2647
3,8dpi,1.0,1387
4,control,1.0,3045


# Visualize cell types

In [34]:
current_index = 4

In [35]:
data = datas[current_index]
#nucleoli_data = datas_nucleoli[current_index]

In [36]:
img = tifffile.imread(data+'Output.tif')
img = img[:,0,:,:]

In [37]:
labels = tifffile.imread(data+'Labels.tif')

In [28]:
tmap = {'0':0, 'Epidermal':1, 'Gut':3, 'Chromatoid':2, 'Secretory':4, 'Unknown':5, 'Distal':6, 'Proximal':7, 'Progentior':8}

In [29]:
correct = True
if correct:
    correction_df = pd.read_csv('Corrections.csv')
    correction_df['Id'] = correction_df['Id'] - 1
    correction_df['Id'] = correction_df['Id'].astype(str).str.zfill(4)
    for i,a in correction_df.iterrows():
        df.loc[(df['Dataset']==a['Dataset']) & (df['Id']==a['Id']), 'Type'] = 'Epidermal'
    

In [38]:
ids = df[df['Dataset']==data_fix[current_index]].sort_values(by='Id')['Type'].map(tmap).fillna(0).values

In [39]:
ids = np.concatenate([[0],ids])

In [40]:
id_img = ids[labels]

In [221]:
viewer.add_labels(id_img[::4,::4,::4].astype(int))
viewer.add_image(img[::4,::4,::4], blending='additive', colormap='gray')
viewer.add_labels(labels[::4,::4,::4])
#viewer.add_image(id_img[::4,::4,::4]==2, blending='additive', colormap='gray', name='Chromatoid')

<Labels layer 'Labels [1]' at 0x1e489922df0>

To spit out tiffs so Steph can render in Blender

In [41]:
for cell_type in ['Epidermal', 'Gut', 'Chromatoid', 'Secretory', 'Unknown', 'Distal', 'Proximal', 'Progentior']:
    tifffile.imsave('Graphs/' + data_fix[current_index]+'_'+cell_type+'.tif', 
                    (id_img[::2,::2,::2]==tmap[cell_type]).astype(np.ubyte), imagej=True)

C:\Users\smc\AppData\Local\Temp\ipykernel_18656\2365819362.py:2: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave('Graphs/' + data_fix[current_index]+'_'+cell_type+'.tif',


In [225]:
id_img.shape

(1700, 3360, 3024)

In [54]:
viewer.add_labels(id_img[::4,::4,::4]==2, blending='additive', name='Chromatoid')
viewer.add_labels(id_img[::4,::4,::4]==1, blending='additive', name='Epidermal')
viewer.add_labels(id_img[::4,::4,::4]==4, blending='additive', name='Secretory')
viewer.add_labels(id_img[::4,::4,::4]==3, blending='additive', name='Gut')

<Labels layer 'Gut' at 0x16f777d3520>

In [55]:
viewer.add_image(id_img[::4,::4,::4]==2, blending='additive', colormap='blue', name='Chromatoid', rendering='iso')
viewer.add_image(id_img[::4,::4,::4]==1, blending='additive', colormap='red', name='Epidermal', rendering='iso')
viewer.add_image(id_img[::4,::4,::4]==4, blending='additive', colormap='green', name='Secretory', rendering='iso')

<Image layer 'Secretory [1]' at 0x16f77947df0>

In [97]:
df.loc[df['Type']=='Doublet', 'Type']='Other'
#df[df['Type']=='Other'].to_csv('Others.csv')
#df[df['Type']!='Other'].to_csv('NotOthers.csv')

In [101]:
adf = adf.merge(adf.groupby(['Sample']).agg({'Annotator_y':len}).reset_index(), on='Sample')
adf['Normalizer'] = 1.0/adf['Annotator_y_y']

KeyError: 'Sample'

In [67]:
adf['Sample'].unique()

array(['control', '4dpi', '8dpi', '14dpi'], dtype=object)

# Prepare volumes and centers of mass

In [77]:
max_count = np.max(labels)

In [78]:
volumes = ndimage.labeled_comprehension(labels, labels, np.arange(1,max_count+1), len, float, 0)

KeyboardInterrupt: 

In [ ]:
centers_of_mass = ndimage.center_of_mass(labels, labels, np.arange(1,max_count+1))

In [ ]:
centers_of_mass = np.array(centers_of_mass)
centers_of_mass.shape

In [ ]:
np.save(data+'CentersMass.npy', centers_of_mass)
np.save(data+'Volumes.npy', volumes)

# Load volumes and centers of mass

In [60]:
df = df.sort_values(by='Id')

In [61]:
def get_dataframe(df, data, nucleoli_data, dataset_in):
    print(data)
    centers_of_mass = np.load(data+'CentersMass.npy')
    volumes = np.load(data+'Volumes.npy')
    nucleoli_labels = np.load(nucleoli_data+'Nucleoli_Counts.npy')
    nucleoli_volumes = np.load(nucleoli_data+'Nucleoli_Volumes.npy')
    cdf = df[df['Dataset']==dataset_in]
    
    cdf['Volumes'] = volumes
    cdf['Nucleoli'] = nucleoli_labels
    cdf['NucleoliVolumes'] = nucleoli_volumes
    
    cdf['X'] = centers_of_mass[:,2]
    cdf['Y'] = centers_of_mass[:,1]
    cdf['Z'] = centers_of_mass[:,0]

    cdf['IntX'] = cdf['X'].astype(int)
    cdf['IntY'] = cdf['Y'].astype(int)
    cdf['IntZ'] = cdf['Z'].astype(int)
    
    return cdf

In [102]:
adf = pd.DataFrame()
for a in range(0,4):
    print(data_fix[a])
    cdf = get_dataframe(df, datas[a], datas_nucleoli[a], data_fix[a])
    cdf['Sample'] = data_fix[a]
    adf = pd.concat([adf, cdf])


control
S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/
4dpi
S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210909_MER_20-15-5B_4dpi/cjw/DeepLearn/
8dpi
S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210129_MER_20-15-7b_8dpi_stack2/cjw/DeepLearn/
14dpi
S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210326_MER_20-15-8a_14dpi/cjw/DeepLearn/


C:\Users\smc\AppData\Local\Temp\ipykernel_15808\2256817690.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\smc\AppData\Local\Temp\ipykernel_15808\2256817690.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\smc\AppData\Local\Temp\ipykernel_15808\2256817690.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [103]:
adf['Sample'].unique()

array(['control', '4dpi', '8dpi', '14dpi'], dtype=object)

In [104]:
df

,Type,File,Annotator,Dataset,Id,,Fraction
0,Chromatoid,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,MEL,control,0579,NaN,0.000328
1,Unknown,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,MOL,control,2590,NaN,0.000328
2,Epidermal,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,MEL,control,1672,NaN,0.000328
3,Chromatoid,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,MEL,control,1840,NaN,0.000328
4,Chromatoid,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,MEL,control,1260,NaN,0.000328
...,...,...,...,...,...,...,...
11080,Chromatoid,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,STN,48dpa,1036,980.0,0.000395
11081,Unknown,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,STN,48dpa,2481,981.0,0.000395
11082,Gut,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,STN,48dpa,0932,982.0,0.000395
11083,Unknown,S:/micro/asa/stn/mel/20-15_Irradiation_Dataset...,STN,48dpa,0337,983.0,0.000395


Load muscle data and merge with control

In [29]:
edt = tifffile.imread('S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn_DV/EDT.tif')

In [44]:
dists = sp.ndimage.labeled_comprehension(edt, labels, np.arange(1,np.max(labels)+1), np.mean, np.float, 0)

C:\Users\smc\AppData\Local\Temp\ipykernel_18580\603810153.py:1: DeprecationWarning:

`np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations



In [45]:
control_df = adf[adf['Dataset']=='control']
control_df['MuscleDistance'] = dists

C:\Users\smc\AppData\Local\Temp\ipykernel_18580\3148024471.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [47]:
f = px.box(control_df, x='Type', y='MuscleDistance',points='all')
f.write_html('MuscleProximity.html')
f

In [58]:
bdf = adf.copy()

bdf.loc[(bdf['Volumes']<250000) & (bdf['Type']=='Chromatoid'), 'Type'] = 'SmallChromatoid'


ids = bdf[bdf['Dataset']==data_fix[current_index]].sort_values(by='Id')['Type'].map({'0':0, 'Epithelial':1, 'Gut':3, 'Chromatoid':2, 'Secretory':4, 'Other':5, 'SmallChromatoid':6}).fillna(0).values
ids = np.concatenate([[0],ids])
id_img = ids[labels]


MemoryError: Unable to allocate 129. GiB for an array with shape (1700, 3360, 3024) and data type float64

In [21]:
downsample = 4

In [22]:
viewer.add_labels(id_img[::downsample,::downsample,::downsample].astype(int))
viewer.add_image(img[::downsample,::downsample,::downsample], blending='additive', colormap='gray')

<Image layer 'Image' at 0x1b312fa7808>

In [23]:
viewer.add_image(id_img[::downsample,::downsample,::downsample]==2, blending='additive', colormap='magenta', name='Chromatoid')
viewer.add_image(id_img[::downsample,::downsample,::downsample]==6, blending='additive', colormap='yellow', name='SmallChromatoid')
viewer.add_image(id_img[::downsample,::downsample,::downsample]==3, blending='additive', colormap='red', name='Gut')
viewer.add_image(id_img[::downsample,::downsample,::downsample]==1, blending='additive', colormap='blue', name='Epidermal')
viewer.add_image(id_img[::downsample,::downsample,::downsample]==4, blending='additive', colormap='cyan', name='Secretory')
viewer.add_image(id_img[::downsample,::downsample,::downsample]==5, blending='additive', colormap='green', name='Other')

<Image layer 'Other' at 0x1b31326b708>

In [85]:
aadf = bdf[bdf['Sample']==data_fix[current_index]]
aadf = aadf[(aadf['Z']>100) & (aadf['Z']<1500)]

In [86]:
px.box(aadf, x='Type', y='Volumes', points='all')

In [87]:
px.box(aadf[aadf['Type'].isin(['SmallChromatoid', 'Chromatoid', 'Gut', 'Epithelial'])], x='Type', y='Volumes', points='all',
      category_orders={'Type':['Epithelial', 'Gut', 'Chromatoid', 'SmallChromatoid']})

In [62]:
px.histogram(aadf[aadf['Type'].isin(['SmallChromatoid', 'Chromatoid', 'Gut', 'Epithelial'])], x='X', facet_row='Type', 
             height=800, nbins=20, category_orders={'Type':['Epithelial', 'Gut', 'Chromatoid', 'SmallChromatoid']})

In [38]:
direct = 'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Graphs/Images/'

for a in viewer.layers:
    tifffile.imsave(direct+a.name+'.tif', a.data.astype(np.ubyte), imagej=True)

## Look at all time points at once

In [90]:
adf = adf.merge(adf.groupby(['Sample']).agg({'Annotator_y':len}).reset_index(), on='Sample')
adf['Normalizer'] = 1.0/adf['Annotator_y_y']

In [105]:
px.bar(adf.groupby(['Type', 'Sample']).agg({'Normalizer':np.sum}).reset_index(),
       x='Sample', y='Normalizer', color='Type', 
       category_orders=cats)

In [103]:
px.bar(adf.groupby(['Type', 'Sample']).agg({'Normalizer':np.sum}).reset_index(),
       x='Sample', y='Normalizer', animation_frame='Type', 
       category_orders=cats)

In [101]:
px.bar(adf.groupby(['Type', 'Sample']).agg({'Normalizer':np.sum}).reset_index(),
       x='Sample', y='Normalizer', facet_col='Type', facet_col_wrap=4, height=1600, 
       category_orders=cats)

In [72]:
adf['NucleoliFraction'] = adf['NucleoliVolumes']/adf['Volumes']
px.box(adf, x='Type', y='NucleoliFraction', points='all', color='Annotator', animation_frame='Dataset', hover_data=['File'],
      category_orders=cats)

ValueError: Value of 'hover_data_0' is not the name of a column in 'data_frame'. Expected one of [' _x', 'Type_x', 'File_x', 'Annotator_x', 'Dataset', 'Id', ' _y', 'Type_y', 'File_y', 'Annotator_y', 'Type', 'Annotator', 'Volumes', 'Nucleoli', 'NucleoliVolumes', 'X', 'Y', 'Z', 'IntX', 'IntY', 'IntZ', 'Sample', 'NucleoliFraction'] but received: File

In [23]:
df['Type'].value_counts()

Other         4304
Epithelial    1444
Gut           1242
Secretory      758
Chromatoid     580
Junk           223
Name: Type, dtype: int64

In [27]:
px.histogram(adf[adf['Type']=='Chromatoid'], x='Volumes', facet_row='Type', height=800, nbins=150)

In [25]:
px.box(adf[adf['Sample']==data_fix[current_index]].reset_index().reset_index(), x='Type', y='NucleoliFraction', hover_data=['IntX', 'IntY', 'IntZ'], 
                         height=800, color='Dataset',
                        points='all')

## Look at only the selected (interactive)

In [56]:
viewer.add_image(img, colormap='gray', blending='additive')
viewer.layers[-1].contrast_limits=[13, 213]
viewer.add_labels(labels)
viewer.add_labels(id_img.astype(int))

MemoryError: Unable to allocate 129. GiB for an array with shape (1700, 3360, 3024) and data type float64

In [59]:
adf.columns

Index([' ', 'Type', 'File', 'Annotator', 'Dataset', 'Id'], dtype='object')

In [ ]:
f=go.FigureWidget(px.box(adf[adf['Sample']==data_fix[current_index]].reset_index().reset_index(), x='Type', y='Volumes', hover_data=['IntX', 'IntY', 'IntZ'], 
                         height=800, color='Annotator',
                        points='all'))

def click_fn(trace, points, state):
    #print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
    if (len(points.point_inds)>0):
        print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
        x = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        y = f.data[points.trace_index]['customdata'][points.point_inds[-1]][1]
        z = f.data[points.trace_index]['customdata'][points.point_inds[-1]][2]
        
        viewer.camera.center = [0,y,x]
        viewer.camera.zoom=4
        viewer.dims.current_step = (z,0,0)
        
     #file = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        #show_image(file+'_processed.tif', viewer)

for s in f.data:
    s.on_click(click_fn)

f

In [55]:
f=go.FigureWidget(px.scatter(cdf.reset_index().reset_index(), x='Volumes', y='NucleoliVolumes', color='Type', hover_data=['IntX', 'IntY', 'IntZ', 'Annotator'], height=800))

def click_fn(trace, points, state):
    #print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
    if (len(points.point_inds)>0):
        print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
        x = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        y = f.data[points.trace_index]['customdata'][points.point_inds[-1]][1]
        z = f.data[points.trace_index]['customdata'][points.point_inds[-1]][2]
        
        viewer.camera.center = [0,y,x]
        viewer.camera.zoom=4
        viewer.dims.current_step = (z,0,0)
        
     #file = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        #show_image(file+'_processed.tif', viewer)

for s in f.data:
    s.on_click(click_fn)

f

FigureWidget({
    'data': [{'customdata': array([[2151, 80, 20, 'STN'],
                                   [4…

## Look at spatial distributions

In [52]:
size = 20

aadf = adf[(adf['Dataset']==data_fix[current_index]) & (adf['Volumes']<250000) & (adf['Type']=='Chromatoid')]
viewer.add_points(np.array([aadf['Z']/2, aadf['Y']/2, aadf['X']/2]).T, face_color='green', size=size, name='Small Chromatoid')

aadf = adf[(adf['Dataset']==data_fix[current_index]) & (adf['Volumes']>250000) & (adf['Type']=='Chromatoid')]
viewer.add_points(np.array([aadf['Z']/2, aadf['Y']/2, aadf['X']/2]).T, face_color='magenta', size=size, name='Large Chromatoid')

aadf = adf[(adf['Dataset']==data_fix[current_index]) & (adf['Volumes']>250) & (adf['Type']=='Gut')]
viewer.add_points(np.array([aadf['Z']/2, aadf['Y']/2, aadf['X']/2]).T, face_color='red', size=size, name='Gut')

aadf = adf[(adf['Dataset']==data_fix[current_index]) & (adf['Volumes']>250) & (adf['Type']=='Epithelial')]
viewer.add_points(np.array([aadf['Z']/2, aadf['Y']/2, aadf['X']/2]).T, face_color='gray', size=size, name='Epithelial')

aadf = adf[(adf['Dataset']==data_fix[current_index]) & (adf['Volumes']>250) & (adf['Type']=='Other')]
viewer.add_points(np.array([aadf['Z']/2, aadf['Y']/2, aadf['X']/2]).T, face_color=[10,10,10], size=size, name='Other')

c:\users\smc\anaconda3\envs\ip\lib\site-packages\napari\utils\colormaps\standardize_color.py:303: UserWarning:

Colors with values larger than one detected. napari will normalize these colors for you. If you'd like to convert these yourself, please use the proper method from skimage.color.



<Points layer 'Other [1]' at 0x1c60b8befc8>

In [53]:
lst = ['Gut', 'Chromatoid', 'Other', 'Secretory', 'Epithelial', 'SmallChromatoid']

In [54]:
aadf = adf[adf['Dataset']==data_fix[current_index]]
aadf['Sub-Type'] = aadf['Type']
aadf.loc[(aadf['Type']=='Chromatoid') & (aadf['Volumes']<250000),'Sub-Type'] = 'SmallChromatoid'
for a in lst:
    rslt = sp.spatial.distance.cdist(np.array(aadf[['X', 'Y', 'Z']]),
                                  np.array(aadf[aadf['Sub-Type']==a][['X', 'Y', 'Z']]))
    rslt[rslt==0]=5000
    aadf[a] = np.min(rslt, axis=1)

c:\users\smc\anaconda3\envs\ip\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

c:\users\smc\anaconda3\envs\ip\lib\site-packages\pandas\core\indexing.py:1720: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

c:\users\smc\anaconda3\envs\ip\lib\site-packages\ipykernel_launcher.py:8: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

In [64]:
#px.box(aadf, x='Sub-Type', y='Chromatoid', points='all')

In [56]:
aadf['ClosestType'] = np.array(lst)[np.argmin(aadf[lst].values, axis=1).astype(int)]

c:\users\smc\anaconda3\envs\ip\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [57]:
count_df = aadf.pivot_table(index='Sub-Type', aggfunc=len, values='Y').reset_index()

In [58]:
count_df['Dummy'] = 'Test'

In [59]:
px.bar(count_df, x='Dummy', y='Y', color='Sub-Type',
      category_orders = {'Sub-Type':['Other', 'Chromatoid',  'SmallChromatoid', 'Secretory', 'Gut', 'Epithelial', 'Junk', 'Doublet'],
                        'ClosestType':['Other', 'Chromatoid',  'SmallChromatoid', 'Secretory', 'Gut', 'Epithelial', 'Junk', 'Doublet']})

In [60]:
aadf = aadf.merge(count_df, on='Sub-Type')
aadf['Unit'] = 1.0/aadf['Y_y']

In [61]:
px.bar(aadf.groupby(['Sub-Type', 'ClosestType']).agg({'Unit':np.sum}).reset_index(), x='Sub-Type', y='Unit', color='ClosestType',
      category_orders = {'Sub-Type':['Other', 'Chromatoid',  'SmallChromatoid', 'Secretory', 'Gut', 'Epithelial', 'Junk', 'Doublet'],
                        'ClosestType':['Other', 'Chromatoid',  'SmallChromatoid', 'Secretory', 'Gut', 'Epithelial', 'Junk', 'Doublet']},
      title='Nearest Neighbor Probabilities')

In [106]:
viewer.add_image(img, colormap='gray', blending='additive')
viewer.layers[-1].contrast_limits=[13, 213]
viewer.add_labels(labels)
viewer.add_labels(id_img.astype(int))

<Labels layer 'Labels' at 0x20d10bef188>

In [110]:
f=go.FigureWidget(px.box(aadf, x='Sub-Type', y='Chromatoid', hover_data=['IntX', 'IntY', 'IntZ'], 
                         height=800, 
                        points='all'))

def click_fn(trace, points, state):
    #print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
    if (len(points.point_inds)>0):
        print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
        x = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        y = f.data[points.trace_index]['customdata'][points.point_inds[-1]][1]
        z = f.data[points.trace_index]['customdata'][points.point_inds[-1]][2]
        
        viewer.camera.center = [0,y,x]
        viewer.camera.zoom=4
        viewer.dims.current_step = (z,0,0)
        
     #file = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        #show_image(file+'_processed.tif', viewer)

for s in f.data:
    s.on_click(click_fn)

f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'custom…

[1684  565  163]


In [87]:
np.min(rslt, axis=1).

(3045,)

## Nucleoli counts in control (not used anymore)

In [ ]:
nucleoli_df = pd.read_csv('Classifier/Montage_List_STN_20220418_final.csv')

In [ ]:
nucleoli_df['Dataset'] = nucleoli_df['File'].str.split('_').str[-1].str.split('.').str[0]
nucleoli_df['Id'] = nucleoli_df['File'].str.split('_').str[-2].str.split('/').str[-1]
nucleoli_df = nucleoli_df.rename(columns={'Type':'Nucleoli'})
nuc_mapper = {'Secretory':1, 'Junk':6, 'Doublet':2, 'Neuronal':0, 'Muscle':3}
nucleoli_df['Nucleoli'] = nucleoli_df['Nucleoli'].map(nuc_mapper)

In [ ]:
nucleoli_df = nucleoli_df.merge(df, on=['Dataset', 'Id'])

In [ ]:
nucleoli_df = nucleoli_df[nucleoli_df['Nucleoli']<5]

In [ ]:
nucleoli_pivot = nucleoli_df.pivot_table(index='Type', values='Nucleoli', aggfunc=[np.mean, sp.stats.sem]).reset_index().sort_values(by=('mean', 'Nucleoli'), ascending=False)

C:\Users\smc\Miniconda3\envs\napari\lib\site-packages\numpy\core\_methods.py:265: RuntimeWarning:

Degrees of freedom <= 0 for slice

C:\Users\smc\Miniconda3\envs\napari\lib\site-packages\numpy\core\_methods.py:257: RuntimeWarning:

invalid value encountered in double_scalars



In [ ]:
nucleoli_pivot.columns = nucleoli_pivot.columns.map(''.join)

In [ ]:
nucleoli_pivot

,Type,meanNucleoli,semNucleoli
1,ChromatoidDoublet,2.000000,NaN
0,Chromatoid,1.769231,0.052009
7,Progenitor,1.666667,0.235702
9,Secretory,1.358974,0.077818
8,Proximal,1.307692,0.133235
10,Unknown,1.262069,0.036647
3,Doublet,1.200000,0.200000
5,Gut,1.088235,0.034652
2,Distal,1.000000,NaN
6,Muscle,1.000000,0.000000


In [ ]:
f = px.bar(nucleoli_pivot[nucleoli_pivot['Type'].isin(['Chromatoid', 'Progenitor', 
                                                   'Secretory', 'Unknown', 'Gut', 'Epidermal'])], 
       x='Type', y='meanNucleoli', error_y='semNucleoli', title='Nucleoli per cell by cell type')
f.write_html('NucleoliByCellType.html')
f

Load nucleoli

In [14]:
nucleoli_labels = np.load(nucleoli_data+'Nucleoli_Counts.npy')

In [15]:
nucleoli_labels = np.concatenate([[0], nucleoli_labels])

In [16]:
nucleoli_img = nucleoli_labels[labels]

In [17]:
viewer.add_labels(nucleoli_img[::2,::2,::2].astype(int))

<Labels layer 'Labels [1]' at 0x24903fc0c08>

In [18]:
viewer.add_image(img[::2,::2,::2], blending='additive', colormap='gray')

<Image layer 'Image' at 0x24905afda08>

In [44]:
mask = (df[(df['Dataset']=='control')].sort_values(by='Id')['Type'] == 'Epithelial').values

In [45]:
mask = np.concatenate([[False], mask])

In [11]:
epis = mask.astype(int)[labels]

NameError: name 'mask' is not defined